[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day2_Hydroeconomics_and_Water_Mapping/04_platforms_gee_colab_and_qgis_es.ipynb)

# Día 2 - Hidroeconomía y Mapeo del Agua
## Sesión 4: Plataformas - Google Earth Engine, Colab y QGIS

| | |
|---|---|
| **Fecha** | 2026-10-13 (martes) |
| **Horario** | 15:40 - 17:10 |
| **Entidad** | SEI / IMDEA |
| **Instructores** | Christhian, Sebastian |

### Contenido

> Plataformas, cuentas e instalación. Registro en GEE Community, proyecto de Cloud, autenticación de la API, primeros comandos. Google Colab, QGIS 3.44 LTR y repositorio compartido de scripts

### Objetivos de aprendizaje

- [ ] Describir qué es Earth Engine, dónde se realiza el cálculo y qué hace la librería cliente
- [ ] Registrar una cuenta, crear un proyecto de Cloud y autenticar la API de Python
- [ ] Cargar e inspeccionar una `ee.Image` (NASADEM) y una `ee.ImageCollection` (Landsat 9)
- [ ] Definir un área de interés de tres formas - un punto, una geometría dibujada en el mapa y un asset `FeatureCollection` - y visualizar el resultado
- [ ] Exportar una imagen directamente a la computadora y a Google Drive con una tarea de exportación

### Datos necesarios

Una cuenta de Earth Engine con un proyecto de Cloud (sección 4.2). Todos los datos se leen del
catálogo, excepto el asset vectorial usado en la sección 4.7:
`projects/ee-christhianscunha/assets/transcend/basin_Titicaca`.

---


## 4.1 - Qué es Earth Engine

Google Earth Engine es una plataforma en la nube para el análisis geoespacial a escala planetaria. Combina un
catálogo de varios petabytes de imágenes satelitales y datos ambientales con la capacidad de
cálculo necesaria para analizarlos, de modo que el trabajo de almacenar, georreferenciar y preprocesar
el archivo se hace una sola vez, de forma centralizada, en lugar de que lo haga cada investigador que lo necesita. La
plataforma fue descrita por sus creadores en
[Gorelick et al. (2017)](https://doi.org/10.1016/j.rse.2017.06.031), *Google Earth Engine:
Planetary-scale geospatial analysis for everyone*, Remote Sensing of Environment 202,
18-27 - que sigue siendo la referencia estándar para citar cuando se publica un trabajo hecho con ella. La idea
en una línea: llevar el algoritmo a los datos, no los datos al algoritmo.

<style>
img[alt="image-2.png"] { width: 500px; }
</style>
![image-2.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-2.png)

Google Earth Engine tiene dos partes que es fácil confundir:

1. **Un catálogo de datos.** Varios petabytes de imágenes satelitales y productos geoespaciales,
   ya georreferenciados, calibrados y actualizados - el archivo completo de Landsat desde
   1972, Sentinel-1 y -2, MODIS, reanálisis climáticos, modelos digitales de elevación, productos de
   cobertura del suelo. No es necesario descargar ni preprocesar los datos.
2. **Un servicio de procesamiento.** El cálculo se ejecuta en la infraestructura de Google, cerca de los
   datos, y se distribuye entre muchas máquinas.

3. **No es lo mismo que Google Earth.** A pesar del nombre compartido, Earth Engine no es el
 globo que se hace girar en Google Earth. Google Earth es un visor: muestra un mosaico ya hecho y
mejorado visualmente para que usted lo mire. Earth Engine le da los píxeles originales, de calidad
científica - valores de reflectancia, temperaturas de brillo, retrodispersión de
radar - con sus bandas, fechas y metadatos originales intactos, y le permite
hacer cálculos con ellos. En Google Earth usted *mira* la Tierra; en Earth Engine usted la *mide*.

<style>
img[alt="image.png"] { width: 500px; }
</style>

![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image.png)

### Los objetos con los que va a trabajar

Earth Engine ofrece un pequeño conjunto de tipos del lado del servidor, y casi todo script es una
combinación de ellos. `Image` es el ráster básico - una pila de bandas con nombre - e
`ImageCollection` es una serie temporal de esas imágenes. En el lado vectorial, `Geometry` es una
forma, `Feature` es una geometría con atributos y `FeatureCollection` es un conjunto de
features, más o menos un shapefile que vive en el servidor. `Reducer` es lo que reduce muchos valores
a menos: la media de una región, la mediana de una colección, un histograma. `Join` combina
dos colecciones por tiempo, ubicación o una propiedad compartida, mientras que `Array` maneja
cálculos multidimensionales y `Chart` convierte las reducciones en gráficos. Los capítulos siguientes usan
sobre todo los seis primeros; `Array` y `Chart` aparecen solo donde se necesitan.

<style>
img[alt^="Screenshot"] { width: 500px; display: block; margin-bottom: 12px; }
</style>

![Screenshot 2026-09-21 111130.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_Screenshot_2026-09-21_111130.png)
![Screenshot 2026-09-21 111233.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_Screenshot_2026-09-21_111233.png)

### Dónde se ejecuta el código

El catálogo y los servidores son los mismos para todos; lo que cambia es el cliente que
se usa para enviarles instrucciones. Hay cuatro de uso común.

El **Code Editor** (`code.earthengine.google.com`) es un IDE web con una API de JavaScript,
un panel de mapa, un inspector para consultar píxeles, un administrador de tareas para las exportaciones y
la posibilidad de compartir scripts. No hay nada que instalar, y es donde están escritos la documentación oficial y la mayoría de los
ejemplos en línea - lo que lo convierte en el lugar más rápido para crear prototipos y para leer
el código de otras personas.
<style>
img[alt="image-3.png"] { width: 600px; }
</style>
![image-3.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-3.png)

La **API de Python** (`earthengine-api`, importada como `ee`) es la misma API en un notebook o
script, y la razón por la que la mayor parte del trabajo científico se hace aquí: el resultado de un análisis queda
junto a pandas, xarray, scikit-learn y su librería de gráficos, y todo el flujo de trabajo sigue siendo
reproducible en un solo archivo. No incluye un mapa propio - `geemap` o `folium` lo
proporcionan. Este es el cliente que se usa en todo este capítulo.

<style>
img[alt="image-5.png"] { width: 300px; }
</style>
![image-5.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-5.png)


**R** y **Julia** también están cubiertos, por los paquetes comunitarios `rgee` y `EarthEngine.jl`.
Ambos son capas sobre la API de Python y no implementaciones independientes, por lo que Python
debe estar instalado por debajo, y ambos son convenientes si el resto de su análisis ya
está en esos lenguajes.

<style>
img[alt="image-4.png"] { width: 300px; }
</style>
![image-4.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-4.png)

Lo importante es que estos son cuatro dialectos de una misma API. Los tipos de objeto son los
mismos, los nombres de los métodos son los mismos, y solo cambia la sintaxis de la llamada -
`ee.Image('...')` en JavaScript y Python, `ee$Image('...')` en R. Un script leído en el
Code Editor se traduce a Python casi línea por línea, y así se aprovecha el gran
conjunto de ejemplos en JavaScript mientras se trabaja en un notebook.

![image-6.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-6.png)

### El catálogo de datos de Earth Engine

Una de las principales ventajas de Earth Engine es su catálogo público de datos: varios petabytes
de imágenes satelitales y conjuntos de datos geoespaciales, ya georreferenciados, preprocesados y listos
para analizar. Los datos están en los servidores de Google, así que no hay nada que descargar ni almacenar
localmente - simplemente se llama a un conjunto de datos por su ID (por ejemplo `NASA/NASADEM_HGT/001`) y el
cálculo se ejecuta en la nube. Solo el resultado final necesita exportarse, si es que hace falta.

El catálogo incluye, entre otros:

- **Imágenes ópticas** - el archivo completo de Landsat (de 1972 a la actualidad), Sentinel-2 y MODIS.
- **Radar** - Sentinel-1 SAR, que ve a través de las nubes y funciona de día y de noche.
- **Elevación** - NASADEM, SRTM, Copernicus DEM y ALOS.
- **Clima y tiempo** - precipitación (CHIRPS, GPM), reanálisis (ERA5-Land) y
  TerraClimate.
- **Agua** - el conjunto de datos JRC Global Surface Water, con la historia de la ocurrencia de agua
  desde 1984.
- **Cobertura del suelo** - ESA WorldCover, Dynamic World y MapBiomas.
- **Población y límites** - WorldPop, GHSL y límites administrativos como FAO GAUL.

La mayoría de los conjuntos de datos se actualizan regularmente, y las nuevas escenas aparecen pocos días después de su adquisición.
Cada conjunto de datos tiene su propia página que describe las bandas, los factores de escala, la resolución espacial y temporal,
la licencia y un ejemplo de código - vale la pena leerla antes de usar cualquier producto.

Explore el catálogo aquí: <https://developers.google.com/earth-engine/datasets?hl=es-419>

<p align="center">
  <img src="https://developers.google.com/static/earth-engine/images/datasets/gifs/nasa_nex_07_2018.gif" width="32%">
  <img src="https://developers.google.com/static/earth-engine/images/datasets/gifs/modis_006_mod09ga.gif" width="32%">
  <img src="https://developers.google.com/static/earth-engine/images/datasets/gifs/dynamic_world_bissau.gif" width="32%">
</p>

Esta es la parte que cambia la forma de escribir código. Cuando usted escribe:

```python
dem = ee.Image("NASA/NASADEM_HGT/001")
```

no se carga ningún ráster en su computadora. `dem` es una **referencia** (*handle*): la descripción de un objeto
que vive en el servidor. Lo mismo vale para cada operación que se encadena sobre él - filtros,
aritmética de bandas, reductores. La librería cliente construye una descripción del cálculo y
la envía al servidor; el servidor la ejecuta y devuelve solo el resultado.


Consecuencias que conviene tener en cuenta:

- **La evaluación es diferida.** No se calcula nada hasta que se pide un resultado. Las tres
  formas de pedirlo son `getInfo()` (devuelve un valor a Python), mostrar una capa en un mapa (devuelve
  teselas de imagen solo para la vista actual) y `Export` (escribe en Drive o en un Asset).
- **Los objetos del servidor y los objetos de Python no se mezclan.** `ee.Number(3).add(4)` se calcula en el
  servidor; `3 + 4` se calcula localmente. Mezclarlos - por ejemplo usar un `if` de Python sobre un
  valor del lado del servidor - no funciona como se espera, porque el valor del servidor no se conoce
  localmente hasta que se llama a `getInfo()`.
- **`getInfo()` es la operación costosa.** Bloquea la ejecución hasta que el servidor termina y transfiere
  datos por la red. Llamarla dentro de un bucle es la causa más común de notebooks
  lentos.

### Para qué se usa, y para qué no

Earth Engine está diseñado para análisis sobre áreas grandes o series temporales largas, donde el costo
de descargar y preprocesar los datos dominaría el trabajo: cambios de cobertura del suelo a lo largo de
varias décadas, dinámica del agua superficial en una cuenca, monitoreo de cultivos en una región.

<style>
img[alt="image-7.png"] { width: 500px; }
</style>
![image-7.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-7.png)

<style> img[alt="image-8.png"] { width: 500px; }
</style>
![image-8.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/04_c02_image-8.png)

Ejemplos: Mapbiomas Perú y Mapbiomas Bolivia

No reemplaza a un SIG de escritorio. La edición de vectores, el diseño cartográfico y los
flujos de trabajo que necesitan una librería local específica se siguen haciendo en QGIS o en Python fuera de
Earth Engine. Los cálculos interactivos también tienen límites de tiempo y de memoria: el trabajo que
los excede tiene que pasarse a una tarea `Export` por lotes.


---

## 4.2 - Cuenta, proyecto de Cloud y autenticación

Earth Engine es gratuito para investigación, educación y uso no comercial, pero el acceso está vinculado a
un proyecto de Google Cloud. Haga esto una sola vez, antes de ejecutar cualquier otra cosa.

### Registro

1. Inicie sesión con una cuenta de Google.
2. Vaya a **<https://code.earthengine.google.com/register>**.
3. Elija **Use with a Cloud Project**, luego **Unpaid usage** y el propósito que corresponda
   (*Academia & Research*, para este curso).
4. Cree un nuevo proyecto de Cloud o seleccione uno existente. Anote el **id del proyecto** - una
   cadena como `ee-yourname`. Es lo que necesita la API de Python.
5. Abra el Code Editor en <https://code.earthengine.google.com/> para confirmar que la cuenta
   funciona. La aprobación del registro puede tardar desde unos minutos hasta un par de días.

### Las dos funciones

- **`ee.Authenticate()`** obtiene las credenciales. Abre una ventana del navegador (o muestra un
  enlace) donde usted concede el acceso, y luego guarda un token en la máquina. Solo es necesario ejecutarla
  de nuevo cuando el token expira o en una máquina nueva.
- **`ee.Initialize(project="...")`** inicia la sesión y la vincula al proyecto de Cloud.
  Tiene que ejecutarse en cada sesión, y es donde se declara el id del proyecto.

### Dónde se ejecuta este notebook

- **Google Colab** - no hay que instalar nada excepto `geemap`; la autenticación se hace en el
  navegador. El entorno de ejecución se descarta al terminar la sesión, por lo que `ee.Authenticate()` se ejecuta de nuevo
  cada vez.
- **Localmente (Jupyter, VS Code)** - instale `earthengine-api` y `geemap` una vez; el token
  se conserva entre sesiones, así que normalmente basta con `ee.Initialize()`.


In [ ]:
# ---------------------------------------------------------------------------
# Librerías (descomente la línea de pip en un entorno nuevo, p. ej. Google Colab)
# ---------------------------------------------------------------------------
#%pip install -q earthengine-api geemap

import ee
import geemap

print("ee", ee.__version__, "| geemap", geemap.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Autenticar e inicializar
# ---------------------------------------------------------------------------
# >>> REEMPLAZAR por el id de su propio proyecto de Cloud <<<
EE_PROJECT = "ee-christhianscunha" #es necesario crear un proyecto en Google Cloud y habilitar en él la API de Earth Engine

ee.Authenticate()                     # guarda las credenciales en esta máquina
ee.Initialize(project=EE_PROJECT)     # inicia la sesión

# Una operación corta del lado del servidor confirma que la conexión funciona
print("Server test:", ee.Number(21).multiply(2).getInfo(), "(expected 42)")
print("Connected to project:", EE_PROJECT)

---

## 4.3 - Abrir una sola imagen: `ee.Image`

Una `ee.Image` es un ráster con una o más **bandas**, más metadatos (`properties`). Aquí
usamos **NASADEM**, un reprocesamiento de los datos de elevación SRTM a 30 m, distribuido como una
sola imagen global.

Los métodos siguientes devuelven todos objetos del lado del servidor; `getInfo()` es lo que trae un valor
a Python. Observe que la imagen es global: pedir sus metadatos es barato, pero pedir
sus píxeles sin restringir el área no lo es.


In [ ]:
# ---------------------------------------------------------------------------
# ee.Image - NASADEM (modelo digital de elevación global, 30 m)
# ---------------------------------------------------------------------------
dem = ee.Image("NASA/NASADEM_HGT/001")

# Todavía no se ha calculado nada - 'dem' es la descripción de un objeto del lado del servidor.
# getInfo() es lo que realmente pide un valor al servidor.
print("Bands       :", dem.bandNames().getInfo())
print("Native scale:", dem.select("elevation").projection().nominalScale().getInfo(), "m")
print("Type        :", type(dem))

---

## 4.4 - Abrir una colección de imágenes: `ee.ImageCollection`

Una `ee.ImageCollection` es una pila de imágenes que comparten la misma estructura - normalmente todas las
escenas que ha adquirido un sensor. Aquí usamos **Landsat 9** Colección 2 Nivel 2 (reflectancia
de superficie), disponible desde 2021.

Una colección no es útil hasta que se filtra. Casi siempre se aplican tres filtros:

| Filtro | Propósito |
|---|---|
| `filterBounds(geometry)` | conserva las escenas que intersectan un área |
| `filterDate(start, end)` | conserva las escenas de un período |
| `filter(ee.Filter.lt(...))` | conserva las escenas que cumplen una condición de metadatos, como la cobertura de nubes |

Las bandas se almacenan como enteros y tienen que reescalarse con los factores de la Colección 2 Nivel 2:

| Bandas | Variable | Escala |
|---|---|---|
| `SR_B1` - `SR_B7` | reflectancia de superficie (0-1) | `0.0000275 * DN - 0.2` |
| `ST_B10` | temperatura de superficie (Kelvin) | `0.00341802 * DN + 149.0` |

La función siguiente aplica ambos y se reutiliza en los próximos ejemplos.

In [ ]:
# ---------------------------------------------------------------------------
# ee.ImageCollection - Landsat 9 Colección 2 Nivel 2
# ---------------------------------------------------------------------------
l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

# Nombres de las bandas de una imagen, para ver qué contiene la colección.
# limit(1) evita recorrer todo el archivo solo para ver la estructura.
print("Bands:", l9.limit(1).first().bandNames().getInfo())

# Parámetros compartidos para los dos ejemplos siguientes
START, END = "2025-09-01", "2026-09-01"
CLOUD_MAX = 30

def scale_l9(image):
    # Factores de escala de la Colección 2 Nivel 2
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)      # reflectancia (0-1)
    thermal = image.select("ST_B10").multiply(0.00341802).add(149.0)   # temperatura (K)
    scaled = optical.addBands(thermal)
    return ee.Image(scaled.copyProperties(image, ["system:time_start", "CLOUD_COVER"]))

# Parámetros de visualización, reutilizados por ambos mapas
vis_rgb = {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 0, "max": 0.3}   # color natural
vis_nir = {"bands": ["SR_B5", "SR_B4", "SR_B3"], "min": 0, "max": 0.3}   # falso color

print("Period:", START, "to", END, "| cloud cover <", CLOUD_MAX, "%")

---

## 4.5 - Ejemplo 1: un área de interés definida por un punto

La forma más simple de ubicar un análisis es un par de coordenadas. `ee.Geometry.Point` las recibe
en el orden **[longitud, latitud]** - el inverso del orden [lat, lon] usado por la mayoría de las
librerías de mapas, incluido `geemap.Map(center=...)`. Intercambiarlas es un error frecuente, y
normalmente se manifiesta como una colección vacía.

Un punto no tiene área, por lo que no se puede usar para recortar una imagen, pero es suficiente para
`filterBounds()`: el filtro conserva todas las escenas cuya huella contiene ese punto.


In [ ]:
# ---------------------------------------------------------------------------
# Ejemplo 1 - punto: [longitud, latitud]
# ---------------------------------------------------------------------------
point = ee.Geometry.Point([-68.7849, -16.5523])   # Altiplano boliviano

l9_point = (l9.filterBounds(point)
              .filterDate(START, END)
              .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_MAX))
              .map(scale_l9))

image_point = ee.Image(l9_point.sort("CLOUD_COVER").first())

print("Scenes containing the point:", l9_point.size().getInfo())
print("Least cloudy scene         :",
      ee.Date(image_point.get("system:time_start")).format("YYYY-MM-dd").getInfo(),
      "|", round(image_point.get("CLOUD_COVER").getInfo(), 1), "% cloud")

Map_point = geemap.Map(center=[-16.5523, -68.7849], zoom=10)   # nota: aquí es [lat, lon]
Map_point.addLayer(image_point, vis_rgb, "Landsat 9 - natural colour")
Map_point.addLayer(point, {"color": "red"}, "Point")

Map_point

---

## 4.6 - Ejemplo 2: un área de interés dibujada en el mapa

Las herramientas de dibujo de la barra de la izquierda producen directamente geometrías de Earth Engine.
`Map.user_roi` devuelve la última geometría dibujada, y `Map.user_rois` devuelve todas ellas como una
`FeatureCollection`. Si no se ha dibujado nada, `user_roi` es `None`.

Dibuje un **rectángulo o un polígono** en el mapa de abajo y luego ejecute la celda siguiente. A diferencia de un punto,
un polígono tiene área, por lo que también se puede usar para **recortar** la imagen - que es lo que mantiene la
visualización restringida al área de interés.


In [ ]:
# ---------------------------------------------------------------------------
# Ejemplo 2 - dibuje un rectángulo o un polígono en este mapa y luego ejecute la celda siguiente
# ---------------------------------------------------------------------------
Map_roi = geemap.Map(center=[-16.5523, -68.7849], zoom=10)
Map_roi.add_basemap("SATELLITE")

Map_roi

In [ ]:
# ---------------------------------------------------------------------------
# Filtrar y recortar con la geometría dibujada
# ---------------------------------------------------------------------------
roi = Map_roi.user_roi if Map_roi.user_roi is not None else point.buffer(15000).bounds()

l9_roi = (l9.filterBounds(roi)
            .filterDate(START, END)
            .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_MAX))
            .map(scale_l9))

# median() combina las escenas en una sola imagen; clip() la restringe a la geometría
image_roi = l9_roi.median().clip(roi)

print("Scenes intersecting the area:", l9_roi.size().getInfo())
print("Area (km2)                  :",
      round(roi.area(maxError=1).divide(1e6).getInfo(), 1))

Map_roi.addLayer(image_roi, vis_rgb, "Landsat 9 - median, clipped")
Map_roi.addLayer(image_roi, vis_nir, "Landsat 9 - false colour", False)
Map_roi.addLayer(roi, {"color": "yellow"}, "Drawn area")
Map_roi.centerObject(roi, 11)

Map_roi

---

## 4.7 - Ejemplo 3: un área de interés a partir de un asset `FeatureCollection`

Un punto y un polígono dibujado se definen en el notebook. En la práctica, el área de interés
normalmente ya existe como una capa vectorial - el límite de una cuenca, un conjunto de parcelas, una
división administrativa. Subirla una vez como **asset de Earth Engine** la deja disponible
para todos los scripts, y para cualquier persona con quien se comparta el asset.

Un asset se referencia por su ruta completa:

```
projects/<project-id>/assets/<folder>/<name>
```

El asset usado aquí, `projects/ee-christhianscunha/assets/transcend/basin_Titicaca`, es una
`FeatureCollection`: un conjunto de features, cada uno con una geometría y una tabla de atributos.

Se muestran dos operaciones:

- **mostrar la capa vectorial**, usando `style()` para dibujar solo los contornos, de modo que el ráster
  de abajo siga visible;
- **recortar la imagen de elevación NASADEM** con `clip(fc)`, que restringe el ráster al
  límite de la colección.

`clip()` solo afecta lo que se muestra y se calcula dentro de la geometría; no
remuestrea ni modifica los píxeles.


In [ ]:
# ---------------------------------------------------------------------------
# Ejemplo 3 - asset FeatureCollection
# ---------------------------------------------------------------------------
tdp = ee.FeatureCollection("projects/ee-christhianscunha/assets/transcend/basin_Titicaca")

print("Features  :", tdp.size().getInfo())
print("Attributes:", tdp.first().propertyNames().getInfo())

In [ ]:
# ---------------------------------------------------------------------------
# Elevación NASADEM recortada por la FeatureCollection
# ---------------------------------------------------------------------------
elevation = dem.select("elevation").clip(tdp)

# Ajustar la escala de colores al rango real del área, en lugar de adivinarlo.
# scale=90 mantiene rápido el cálculo de la estadística; los datos siguen estando a 30 m.
stats = elevation.reduceRegion(reducer=ee.Reducer.minMax(), geometry=tdp.geometry(),
                               scale=90, maxPixels=1e9).getInfo()

vis_dem = {"min": stats["elevation_min"], "max": stats["elevation_max"],
           "palette": ["#f4f1e8", "#cfd8b4", "#9aae7d", "#5c7a54", "#26402f"]}

print("Elevation range: {} to {} m".format(stats["elevation_min"], stats["elevation_max"]))

Map_fc = geemap.Map()
Map_fc.addLayer(elevation, vis_dem, "NASADEM elevation (clipped)")
Map_fc.addLayer(tdp.style(color="#e34948", fillColor="00000000", width=2), {}, "basin_Titicaca")
Map_fc.centerObject(tdp)

Map_fc

### Comparación de los tres enfoques

| | Definido por | Filtra una colección | Recorta un ráster | Reutilizable |
|---|---|---|---|---|
| **Punto** | coordenadas en el código | sí | no (no tiene área) | sí, si se registran las coordenadas |
| **Geometría dibujada** | interacción con el mapa | sí | sí | no - se pierde al reiniciar el notebook |
| **Asset** | un archivo vectorial subido una vez | sí | sí | sí, entre scripts y usuarios |

Una geometría dibujada es práctica para explorar y para enseñar, pero hace que un análisis sea
imposible de reproducir: nadie más puede volver a dibujar el mismo polígono. El trabajo que tiene que
repetirse o compartirse debe usar un asset, o coordenadas escritas explícitamente en el código.


---

## 4.8 - Descargar los resultados

Para abrir los resultados en QGIS, las imágenes tienen que exportarse como archivos GeoTIFF. Dos formas:

- **Descarga directa** con `geemap.ee_export_image()` - el archivo se guarda en su computadora. Solo para archivos pequeños.
- **Tarea de exportación** con `ee.batch.Export.image.toDrive()` - Earth Engine la procesa en segundo plano y guarda el archivo en su Google Drive.

### 4.8.1 - NASADEM: descarga directa a la computadora

Descargamos `elevation`, el NASADEM recortado por `basin_Titicaca` (sección 4.7).

In [ ]:
# ---------------------------------------------------------------------------
# NASADEM recortado por basin_Titicaca -> GeoTIFF en esta computadora
# ---------------------------------------------------------------------------
geemap.ee_export_image(
    elevation,                        # de la sección 4.7
    filename="nasadem_tdp.tif",
    scale=250,                        # tamaño del píxel (m); la cuenca es demasiado grande para una descarga directa a 30 m
    region=tdp.geometry(),
)

# En Google Colab: enviar el archivo a la carpeta de Descargas de su navegador
try:
    from google.colab import files
    files.download("nasadem_tdp.tif")
except ImportError:
    pass

### 4.8.2 - Landsat 9: tarea de exportación a Google Drive

Exportamos `image_roi`, el compuesto de mediana de Landsat 9 de la sección 4.6, con las bandas
ópticas y la banda térmica. El archivo aparece en la carpeta `TRANSCEND_GEE` de su Drive
después de unos minutos; el progreso se puede seguir en <https://code.earthengine.google.com/tasks>.

In [ ]:
# ---------------------------------------------------------------------------
# Compuesto Landsat 9 (óptico + térmico) -> Google Drive
# ---------------------------------------------------------------------------
bands = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10"]

task = ee.batch.Export.image.toDrive(
    image=image_roi.select(bands).toFloat(),   # de la sección 4.6
    description="L9_median_roi",
    folder="TRANSCEND_GEE",
    region=roi,
    scale=30,
    crs="EPSG:32719",                          # UTM 19S (altiplano boliviano)
    maxPixels=1e10,
)
task.start()

print(task.status()["state"])   # ejecute task.status() otra vez más tarde para verificar el progreso

---

## 4.9 - Ejercicio

**Consigna**

1. Elija un área diferente y defínala de las tres formas: un punto con sus coordenadas,
   una geometría dibujada en el mapa y - si tiene uno - un asset vectorial propio.
2. Cargue una colección diferente del catálogo
   (<https://developers.google.com/earth-engine/datasets>) - por ejemplo Sentinel-2
   (`COPERNICUS/S2_SR_HARMONIZED`) o Sentinel-1 (`COPERNICUS/S1_GRD`) - y aplique los
   mismos tres filtros.
3. Exporte el resultado para su área: descargue un recorte pequeño directamente a su computadora y
   envíe la imagen completa a Google Drive como tarea; luego abra ambos archivos en QGIS.
4. Informe, para su área: cuántas escenas devolvió el filtro, la fecha de la escena con menos
   nubes y una captura de pantalla de la visualización.
5. Responda:
   - ¿Qué pasa con el número de escenas si sube el umbral de nubes de 30% a 60%?
   - ¿Qué pasa si invierte el orden de las coordenadas en `ee.Geometry.Point`?
     Explique el resultado.
   - ¿Por qué `filterBounds()` acepta un punto, mientras que `clip()` no produce un resultado
     utilizable con uno?

**Entregable:** el notebook con los tres ejemplos ejecutándose sobre su propia
área, más respuestas cortas por escrito a las preguntas anteriores.


---

## Referencias

**Documentación oficial**

- Registro en Earth Engine: <https://code.earthengine.google.com/register>
- Guías (conceptos, cliente vs servidor, buenas prácticas): <https://developers.google.com/earth-engine/guides>
- **Tutoriales**: <https://developers.google.com/earth-engine/tutorials>
- Catálogo de datos: <https://developers.google.com/earth-engine/datasets>
- Instalación de la API de Python: <https://developers.google.com/earth-engine/guides/python_install>
- Gestión de assets: <https://developers.google.com/earth-engine/guides/asset_manager>

**Libro (acceso abierto)**

- Cardille, J.A., Crowley, M.A., Saah, D., Clinton, N.E. (eds.) (2023).
  *Cloud-Based Remote Sensing with Google Earth Engine: Fundamentals and Applications*.
  Springer. De descarga gratuita, con todos los scripts disponibles.
  - Sitio complementario: <https://www.eefabook.org/>
  - Editorial (acceso abierto): <https://link.springer.com/book/10.1007/978-3-031-26588-4>

**Cursos y material en video**

- **AmbGEO** - formación en geotecnologías y Earth Engine, en portugués, con cursos gratuitos
  y material grabado:
  - <https://ambgeo.com/>
  - YouTube: <https://www.youtube.com/@ambgeo>

**Herramientas de Python**

- Documentación y tutoriales de geemap: <https://geemap.org>
- Wu, Q. (2020). geemap: A Python package for interactive mapping with Google Earth Engine.
  *Journal of Open Source Software*, 5(51), 2305.

**Artículo de referencia**

- Gorelick, N., Hancher, M., Dixon, M., Ilyushchenko, S., Thau, D., Moore, R. (2017).
  Google Earth Engine: Planetary-scale geospatial analysis for everyone.
  *Remote Sensing of Environment*, 202, 18-27.

**Datos usados en este notebook**

- NASADEM: <https://developers.google.com/earth-engine/datasets/catalog/NASA_NASADEM_HGT_001>
- Landsat 9 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2>
